# HR Employee Prediction

This project trains models using five employee features and compares their performance.

The website uses the trained models to predict results for a new employee.

In [ ]:
import os
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Load the dataset
df = pd.read_csv("WA_Fn-UseC_-HR-Employee-Attrition.csv")

# Features used in the project
features = [
    "Age",
    "MonthlyIncome",
    "JobLevel",
    "YearsAtCompany",
    "JobInvolvement"
]

print("Dataset shape:", df.shape)
print("Features:", features)

In [ ]:
# Prepare the targets
df["Attrition_Target"] = df["Attrition"].map({"No": 0, "Yes": 1})
df["OverTime_Target"] = df["OverTime"].map({"No": 0, "Yes": 1})

targets = {
    "Attrition": "Attrition_Target",
    "Overtime": "OverTime_Target",
    "Job Satisfaction": "JobSatisfaction",
    "Performance Rating": "PerformanceRating"
}

data = df[features + list(targets.values())].dropna().copy()

X = data[features]

print("Rows used:", len(data))

In [ ]:
# Models used for comparison
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=8, random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}

def compare_models(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    results = []

    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        results.append({
            "Model": name,
            "Accuracy": accuracy_score(y_test, pred),
            "Precision": precision_score(
                y_test, pred, average="weighted", zero_division=0
            ),
            "Recall": recall_score(
                y_test, pred, average="weighted", zero_division=0
            ),
            "F1-Score": f1_score(
                y_test, pred, average="weighted", zero_division=0
            )
        })

    return pd.DataFrame(results).sort_values(
        "F1-Score", ascending=False
    ).reset_index(drop=True)

In [ ]:
# Compare the models for each prediction
results = {}
best_models = {}

for name, target in targets.items():
    y = data[target].astype(int)

    result = compare_models(X, y)
    results[name] = result

    best_name = result.iloc[0]["Model"]

    # Train the best model on all available training data
    best_model = models[best_name]
    best_model.fit(X, y)

    best_models[name] = best_model

    print("\n" + name)
    print(result.round(3))
    print("Best model:", best_name)

In [ ]:
# Create the models folder
os.makedirs("models", exist_ok=True)

model_files = {
    "Attrition": "final_attrition_model.pkl",
    "Overtime": "final_overtime_model.pkl",
    "Job Satisfaction": "final_satisfaction_model.pkl",
    "Performance Rating": "final_performance_model.pkl"
}

result_files = {
    "Attrition": "attrition_results.csv",
    "Overtime": "overtime_results.csv",
    "Job Satisfaction": "satisfaction_results.csv",
    "Performance Rating": "performance_results.csv"
}

for name in best_models:
    joblib.dump(
        best_models[name],
        os.path.join("models", model_files[name])
    )

    results[name].to_csv(
        os.path.join("models", result_files[name]),
        index=False
    )

joblib.dump(features, "models/feature_columns.pkl")

# Save the data used by the dashboard
data.to_csv(
    "models/employee_analysis_data.csv",
    index=False
)

# Save a summary of the best models
summary = pd.DataFrame({
    "Prediction": list(results.keys()),
    "Best Model": [
        results[name].iloc[0]["Model"] for name in results
    ],
    "Accuracy": [
        results[name].iloc[0]["Accuracy"] for name in results
    ],
    "Precision": [
        results[name].iloc[0]["Precision"] for name in results
    ],
    "Recall": [
        results[name].iloc[0]["Recall"] for name in results
    ],
    "F1-Score": [
        results[name].iloc[0]["F1-Score"] for name in results
    ]
})

summary.to_csv("models/overall_results.csv", index=False)

print("\nModels and results saved successfully.")
print(summary.round(3))

## Test with a new employee

The employee below is created manually. It is not selected from the dataset.

In [ ]:
# New employee example
new_employee = pd.DataFrame([{
    "Age": 28,
    "MonthlyIncome": 4500,
    "JobLevel": 2,
    "YearsAtCompany": 3,
    "JobInvolvement": 3
}])

for name, model in best_models.items():
    prediction = model.predict(new_employee[features])[0]

    if name == "Attrition":
        prediction = "Yes" if prediction == 1 else "No"

    elif name == "Overtime":
        prediction = "Yes" if prediction == 1 else "No"

    print(name + ":", prediction)

In [ ]:
# Check that all files needed by Streamlit exist
files = [
    "final_attrition_model.pkl",
    "final_overtime_model.pkl",
    "final_satisfaction_model.pkl",
    "final_performance_model.pkl",
    "feature_columns.pkl",
    "employee_analysis_data.csv",
    "overall_results.csv",
    "attrition_results.csv",
    "overtime_results.csv",
    "satisfaction_results.csv",
    "performance_results.csv"
]

for file in files:
    print(file, ":", os.path.exists(os.path.join("models", file)))

In [ ]:
# Final feature check
expected_features = ['Age', 'MonthlyIncome', 'JobLevel', 'YearsAtCompany', 'JobInvolvement']
for target, model in best_models.items():
    if hasattr(model, 'feature_names_in_'):
        model_features = list(model.feature_names_in_)
        print(target, model_features)
        if model_features != expected_features:
            raise ValueError(f'Feature mismatch for {target}')
print('All prediction models use the same five features.')


In [ ]:
# Save the exact features used by the website
joblib.dump(expected_features, os.path.join('models', 'feature_columns.pkl'))
print('Saved feature_columns.pkl')
